In [ ]:
from fastai.vision.all import *

In [ ]:
path = untar_data(URLs.IMAGENETTE)

In [ ]:
def get_dls(bs, size):
    return DataBlock(blocks=(ImageBlock(), CategoryBlock()),
                get_items=get_image_files,
                get_y=parent_label,
                item_tfms=Resize(460),
                batch_tfms=[*aug_transforms(size=size, min_scale=0.75),
                            Normalize.from_stats(*imagenet_stats)]
    ).dataloaders(path, bs=bs)

## Progressice Resizing
(trainig the model in multiple steps by changing the image size at each step)

In [ ]:
# Step 1: batch-size=128, image-size=128
dls = get_dls(128, 128)
learn = Learner(
    dls,
    xresnet50(n_out=dls.c),
    loss_func=CrossEntropyLossFlat(),
    metrics=accuracy)

learn.fit_one_cycle(4, 3e-3)

In [ ]:
# Step 2: batch-size=64, image-size=224
learn.dls = get_dls(64, 224)
learn.fit_one_cycle(5, 3e-3)

## Test Time Augmentation (TTA)
(running inference of augumented test image and average results)

In [ ]:
preds,targs = learn.tta()
accuracy(preds, targs).item()

## MixUp
(mix images and labels by random weights, target must be one-hot encoded)

Presudocode:
image2,target2 = dataset[randint(0,len(dataset)]
t = random_float(0.5,1.0)
new_image = t * image1 + (1-t) * image2
new_target = t * target1 + (1-t) * target2

In [ ]:
dls = get_dls(32, 224)
learn = Learner(
    dls,
    xresnet50(n_out=dls.c),
    loss_func=CrossEntropyLossFlat(),
    metrics=accuracy,
    cbs=MixUp())
learn.fit_one_cycle(5, 3e-3)

## Label Smoothing

Change target from (N=10):
[0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
to
[0.01, 0.01, 0.01, 0.91, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01]
Therefore, model must not being over confident.

Replace 0: epsilon/N
Replace 1: 1-epsilon + epsilon/N
Epsilon: 0.1 (we are 10% unsure of our labels)

In [ ]:
dls = get_dls(32, 224)
learn = Learner(
    dls,
    xresnet50(n_out=dls.c),
    loss_func=LabelSmoothingCrossEntropy(),
    metrics=accuracy)
learn.fit_one_cycle(5, 3e-3)